#### Enviroment Setup

In [1]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


#### Loading dataset and filtering it

In [2]:
import pandas as pd
from bertopic import BERTopic

C:\Users\emman\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Reading csv, extracting useful columns, appending future columns
dataset_path = 'dataset/datasetA.csv'
df = pd.read_csv(dataset_path, na_filter=False)


# Code below may or may not be commented out depending on dataset_path
unnecessary_columns = ['date', 'number_of_characters_title', 'number_of_words_title', 'day_of_week', 'month', 'year', 'quarter', 'is_weekend']
df = df.drop(columns=unnecessary_columns)
df[['text', 'authors', 'keywords', 'summary']] = ''

In [ ]:
#Selecting just the title and classes_str columns. Put in a new dataframe so its less confusing for the rest of the code
titles_and_classes = df[['title', 'classes_str']]
#Checking nulls 
titles_and_classes.isna().sum()

In [ ]:
#Filtering the df so it only selects rows where the classes_str value contains 'Learning, Knowledge & Education'
df2 = titles_and_classes[titles_and_classes['classes_str'].str.contains('Learning, Knowledge & Education', case=False)]

#### Testing Various Topic Modeling Techniques

In [ ]:
#BERTopic only reads lists so i did this
docs = df2["title"].tolist()
print(len(docs))

In [ ]:
#Default BERTopic Model 
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)
topic_model.get_topic_info()
"""
Topic is the topic number. -1 refers to outliers so it can be ignored.
Count is how many rows from title fit that topic
Name is the topic name 
Representation is the top words that summarize the topic
Representative_Docs is the example documents that best illustrate the topic"""

In [ ]:
#Work in progress
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
topic_model = BERTopic(embedding_model=embedding_model)
topics, probs = topic_model.fit_transform(docs)
topic_model.get_topic_info()

In [ ]:
#Work in progress
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic(embedding_model=embedding_model)
topics, probs = topic_model.fit_transform(docs)
topics = topic_model.reduce_topics(docs, nr_topics=10)  
topic_model.get_topic_info()

### Webscraping

In [4]:
from newspaper import Article
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\emman\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
# log file for Emmanuel's debuging ventures
import logging
logging.basicConfig(filename='scraperLog.log', format='%(levelname)s: %(asctime)s: %(message)s', datefmt='%m/%d/%Y - %I:%M:%S %p', level=logging.DEBUG)
logging.getLogger('selenium').setLevel(logging.WARNING)
log = logging.getLogger(__name__)
#levels: debug, info, warning, error, critical

In [ ]:
# parses the javascript of the html to redirect and retrieve the correct url
def retrieve_url(driver, link: str, row_index: int = -1, wait_time: int = 1.5) -> (str | None):
    url = None

    try:
        log.debug('POSTing')
        driver.get(link)
        log.debug('POST request completed.')
        
        later = time.time() + wait_time
        while time.time() < later:
            time.sleep(0.01)
        log.debug('Wait time completed.')

        url = driver.execute_cdp_cmd("Runtime.evaluate", {"expression":"location.href"})['result']['value']
        driver.execute_script("window.stop()")
        log.debug('Chrome js script completed.')

        if url.startswith('https://news.google.com') and wait_time < 3:
            newWait = wait_time * 2
            log.info(f'Retrying url for at row {row_index} for {newWait} seconds.')
            url = retrieve_url(driver, link, row_index, newWait)
        
    except Exception as e:
            log.info(f"Couldn't retrieve url at row {row_index}.") if row_index > -1 else log.info("Couldn't retrieve url.")
            log.error(e)

    return url

In [7]:
# return dictionary of authors, keywords, summary, and text
def scrape(url: str, row_index: int = -1) -> (dict[str | list[str]] | None):
    try:
        article = Article(url)
        article.build()
        text = article.text

        if not text:
            row = f' for row {row_index}' if row_index > -1 else ''
            raise Exception(f'Newspaper3k did not return any text{row}.')
        
        return {'authors': article.authors,
                'keywords': article.keywords,
                'summary': article.summary,
                'text': text}
    except Exception as e:
        log.info(f"Failed to scrape article at row {row_index}.")
        log.error(e)
        return

#returns only the text
def scrape_text(url: str, row_index: int = -1) -> (str):
    try:
        article = Article(url)
        article.download()
        article.parse()
        text = article.text
        if not text:
            row = f' for row {row_index}' if row_index > -1 else ''
            raise Exception(f'Newspaper3k did not return any text{row}.')
        return text
    except Exception as e:
        log.info(f"Failed to scrape article at row {row_index}.")
        log.error(e)
        return "Failed to scrape"
    

In [ ]:
#chrome options setup for headless browser and high efficiency
chrome_options = Options()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--disable-renderer-backgrounding")
chrome_options.add_argument("--disable-background-timer-throttling")
chrome_options.add_argument("--disable-backgrounding-occluded-windows")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--disable-cache")
chrome_options.add_argument("--disk-cache-size=0")
chrome_options.add_argument("--media-cache-size=0")
chrome_options.add_argument("--disable-site-isolation-trials")
chrome_options.add_argument("--disable-background-networking")
chrome_options.add_argument("--disable-features=TranslateUI,BackForwardCache")
chrome_options.add_argument("--single-process")
chrome_options.add_argument("disable-sync")
chrome_options.page_load_strategy = "none"

In [ ]:
# input the information and class that you want to scrape, will create a new updated csv
def create_scraped_dataset(required_class: str, filepath: str, NUM_ROWS: int = 10500, BATCH_SIZE: int = 100) -> None:
    with webdriver.Chrome(chrome_options) as driver:
        for row in range(NUM_ROWS):
            ENTRY_IS_BLANK = (not bool(df['text'][row]))

            if required_class.lower() in df['classes_str'][row].lower() and ENTRY_IS_BLANK: 
                link = df['link'][row]
                url = retrieve_url(driver, link, row)

                if url == None:
                    driver.quit()
                    driver = webdriver.Chrome(chrome_options)
                    url = retrieve_url(driver, link, row)

                #url still equal to 'None' after second attempt - pack it up, ggs
                if url == None:
                    data = "Selenium did not handle redirect"
                else:
                    data = scrape_text(url, row)

                df['text'][row] = data
            
            log.debug(f'Row {row} completed.')

            #Process in batches of 100
            if (row + 1) % BATCH_SIZE == 0:
                df.to_csv(filepath, index=False)
                driver.quit()
                driver = webdriver.Chrome(chrome_options)
                log.info(f'{row + 1} rows completed, file saved, driver restarted.')

        df.to_csv(filepath, index=False)
        log.info("Full table walkthrough completed.")
    
    print('something')

In [10]:
CLASS_TO_SCRAPE = 'Learning, Knowledge & Education'
OUTPUT_FILEPATH = 'dataset/newUpdatedDatasetA.csv'
try:
    create_scraped_dataset(CLASS_TO_SCRAPE, OUTPUT_FILEPATH)
except Exception as e:
    log.error(e)

C:\Users\emman\AppData\Roaming\Python\Python314\site-packages\dateutil\parser\_parser.py:1207: UnknownTimezoneWarning: tzname CDT identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "


KeyboardInterrupt: 